# ClipCap Model Integration

Notebook này ghép toàn bộ phần ClipCap đã được nhóm thống nhất:

- CLIP image feature.
- Transformer Mapping Network.
- GPT-2 token embeddings.
- Extended attention mask.
- Extended labels.
- GPT-2 causal language-model loss.
- Gradient từ caption loss về Mapping Network.

Notebook dùng GPT-2 nhỏ được tạo từ config để chạy độc lập và không tải pretrained weights. Đây là bản thử nghiệm trước khi nhóm chuyển thiết kế sang `src/clipcap/models/clipcap_model.py`.

## 1. Pipeline tổng thể

```text
image_embed [B, clip_dim]
        |
        v
TransformerMapper
        |
        v
prefix_embeddings [B, P, D] ------------------+
                                                  |
input_ids [B, L] -> GPT-2 embeddings [B, L, D] |
                                                  v
                              inputs_embeds [B, P + L, D]
                                                  |
attention_mask -> extended_attention_mask -------+
labels         -> extended_labels ---------------+
                                                  v
                                                GPT-2
                                                  |
                                                  v
                                            loss + logits
```

Ký hiệu: `B` là batch size, `P` là prefix length, `L` là caption length và `D` là GPT-2 embedding dimension.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
from torch import Tensor, nn
from transformers import GPT2Config, GPT2LMHeadModel


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.clipcap.models.mapping_network.transformer_mapper import (
        TransformerMapper,
    )
    MAPPER_IMPORT_PATH = "src.clipcap.models.mapping_network"
except ModuleNotFoundError:
    from src.mapping_network.transformer_mapper import TransformerMapper
    MAPPER_IMPORT_PATH = "src.mapping_network"

torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root      :", PROJECT_ROOT)
print("Mapper import path:", MAPPER_IMPORT_PATH)
print("Device            :", DEVICE)
print("PyTorch version   :", torch.__version__)

## 2. Extended attention mask và extended labels

Hai hàm dưới đây giữ đúng contract từ notebook của hai thành viên:

- Prefix attention mask bằng `1` vì GPT-2 phải sử dụng visual prefix làm ngữ cảnh.
- Prefix labels bằng `-100` vì visual prefix không phải token trong vocabulary.
- Padding labels bằng `-100` để không tham gia tính loss.
- Không sửa tensor đầu vào tại chỗ.

In [ ]:
def _validate_prefix_length(prefix_length: int) -> None:
    if (
        isinstance(prefix_length, bool)
        or not isinstance(prefix_length, int)
        or prefix_length <= 0
    ):
        raise ValueError("prefix_length must be a positive integer")


def extend_attention_mask(
    attention_mask: Tensor,
    prefix_length: int,
) -> Tensor:
    if attention_mask.ndim != 2:
        raise ValueError("attention_mask must have shape [B, L]")

    _validate_prefix_length(prefix_length)
    batch_size = attention_mask.size(0)

    prefix_mask = torch.ones(
        (batch_size, prefix_length),
        dtype=attention_mask.dtype,
        device=attention_mask.device,
    )

    return torch.cat([prefix_mask, attention_mask], dim=1)


def build_extended_labels(
    labels: Tensor,
    attention_mask: Tensor,
    prefix_length: int,
) -> Tensor:
    if labels.ndim != 2:
        raise ValueError("labels must have shape [B, L]")
    if attention_mask.ndim != 2:
        raise ValueError("attention_mask must have shape [B, L]")
    if labels.shape != attention_mask.shape:
        raise ValueError("labels and attention_mask must have the same shape")
    if labels.dtype != torch.long:
        raise TypeError("labels must use torch.long dtype")

    _validate_prefix_length(prefix_length)

    caption_labels = labels.clone()
    caption_labels.masked_fill_(attention_mask == 0, -100)

    prefix_labels = torch.full(
        (labels.size(0), prefix_length),
        fill_value=-100,
        dtype=labels.dtype,
        device=labels.device,
    )

    return torch.cat([prefix_labels, caption_labels], dim=1)

## 3. Batch giả và GPT-2 nhỏ

Các kích thước nhỏ giúp notebook chạy nhanh nhưng vẫn kiểm tra đúng interface. Mapping Network và GPT-2 dùng cùng embedding dimension `D`. Caption labels không nằm trong Dataset; khi training, caller truyền `labels=batch["input_ids"]`.

In [ ]:
BATCH_SIZE = 2
CLIP_DIM = 32
EMBEDDING_DIM = 48
CLIP_LENGTH = 3
PREFIX_LENGTH = 5
CAPTION_LENGTH = 6
VOCAB_SIZE = 128

batch = {
    "image_embed": torch.randn(BATCH_SIZE, CLIP_DIM, device=DEVICE),
    "input_ids": torch.tensor(
        [
            [11, 12, 13, 14, 0, 0],
            [21, 22, 23, 24, 25, 0],
        ],
        dtype=torch.long,
        device=DEVICE,
    ),
    "attention_mask": torch.tensor(
        [
            [1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 0],
        ],
        dtype=torch.long,
        device=DEVICE,
    ),
}

assert set(batch) == {"image_embed", "input_ids", "attention_mask"}
assert batch["input_ids"].shape == batch["attention_mask"].shape
assert "labels" not in batch

print("image_embed shape  :", tuple(batch["image_embed"].shape))
print("input_ids shape    :", tuple(batch["input_ids"].shape))
print("attention mask shape:", tuple(batch["attention_mask"].shape))
print("PASS: Dataset contract without labels")

## 4. Thiết kế `ClipCaptionModel`

Model có hai bước rõ ràng:

1. `prepare_gpt2_inputs()` tạo các tensor đã căn chỉnh cho GPT-2.
2. `forward()` gọi GPT-2 và trả output chuẩn của Hugging Face.

Tham số `labels=None` là chủ ý quan trọng:

- Training và validation truyền labels để nhận loss.
- Generation không truyền labels.
- Không dùng `self.training` để quyết định loss, vì validation thường chạy `model.eval()` nhưng vẫn cần loss.

In [ ]:
class ClipCaptionModel(nn.Module):
    def __init__(
        self,
        mapper: TransformerMapper,
        gpt2: GPT2LMHeadModel,
    ) -> None:
        super().__init__()
        self.mapper = mapper
        self.gpt2 = gpt2

        gpt_embedding_dim = self.gpt2.get_input_embeddings().embedding_dim
        if self.mapper.embedding_dim != gpt_embedding_dim:
            raise ValueError(
                "Mapper embedding dimension must match GPT-2 embedding dimension"
            )

    @staticmethod
    def _validate_batch(
        image_embed: Tensor,
        input_ids: Tensor,
        attention_mask: Tensor,
        labels: Tensor | None,
    ) -> None:
        if image_embed.ndim != 2:
            raise ValueError("image_embed must have shape [B, clip_dim]")
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [B, L]")
        if attention_mask.ndim != 2:
            raise ValueError("attention_mask must have shape [B, L]")
        if input_ids.shape != attention_mask.shape:
            raise ValueError("input_ids and attention_mask must have the same shape")
        if image_embed.size(0) != input_ids.size(0):
            raise ValueError("All inputs must have the same batch size")
        if input_ids.dtype != torch.long:
            raise TypeError("input_ids must use torch.long dtype")
        if not (
            image_embed.device
            == input_ids.device
            == attention_mask.device
        ):
            raise ValueError("All inputs must be on the same device")
        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError("labels and input_ids must have the same shape")
            if labels.dtype != torch.long:
                raise TypeError("labels must use torch.long dtype")
            if labels.device != input_ids.device:
                raise ValueError("labels and input_ids must be on the same device")

    def prepare_gpt2_inputs(
        self,
        image_embed: Tensor,
        input_ids: Tensor,
        attention_mask: Tensor,
        labels: Tensor | None = None,
    ) -> dict[str, Tensor | None]:
        self._validate_batch(
            image_embed,
            input_ids,
            attention_mask,
            labels,
        )

        prefix_embeddings = self.mapper(image_embed)
        prefix_length = prefix_embeddings.size(1)
        text_embeddings = self.gpt2.get_input_embeddings()(input_ids)

        if prefix_embeddings.size(2) != text_embeddings.size(2):
            raise ValueError("Prefix and text embedding dimensions do not match")
        if prefix_embeddings.dtype != text_embeddings.dtype:
            raise TypeError("Prefix and text embeddings must use the same dtype")

        inputs_embeds = torch.cat(
            [prefix_embeddings, text_embeddings],
            dim=1,
        )
        if inputs_embeds.size(1) > self.gpt2.config.n_positions:
            raise ValueError("Combined sequence exceeds GPT-2 position limit")

        extended_attention_mask = extend_attention_mask(
            attention_mask,
            prefix_length,
        )

        extended_labels = None
        if labels is not None:
            extended_labels = build_extended_labels(
                labels,
                attention_mask,
                prefix_length,
            )

        sequence_length = inputs_embeds.size(1)
        if extended_attention_mask.size(1) != sequence_length:
            raise ValueError("Extended attention mask length mismatch")
        if (
            extended_labels is not None
            and extended_labels.size(1) != sequence_length
        ):
            raise ValueError("Extended labels length mismatch")

        return {
            "inputs_embeds": inputs_embeds,
            "attention_mask": extended_attention_mask,
            "labels": extended_labels,
        }

    def forward(
        self,
        image_embed: Tensor,
        input_ids: Tensor,
        attention_mask: Tensor,
        labels: Tensor | None = None,
    ):
        gpt2_inputs = self.prepare_gpt2_inputs(
            image_embed=image_embed,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        return self.gpt2(**gpt2_inputs, return_dict=True)

## 5. Khởi tạo Mapper và GPT-2

Không hard-code kích thước CLIP/GPT-2 trong production. Khi dùng pretrained model thật:

```python
clip_dim = int(clip_feature_data["feature_dim"])
embedding_dim = gpt2.get_input_embeddings().embedding_dim
```

Notebook dùng các hằng số nhỏ ở trên chỉ để sanity check.

In [ ]:
mapper = TransformerMapper(
    clip_dim=CLIP_DIM,
    embedding_dim=EMBEDDING_DIM,
    clip_length=CLIP_LENGTH,
    prefix_length=PREFIX_LENGTH,
    num_layers=2,
    num_heads=6,
    feedforward_dim=96,
    dropout=0.0,
)

gpt2_config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=PREFIX_LENGTH + CAPTION_LENGTH + 4,
    n_ctx=PREFIX_LENGTH + CAPTION_LENGTH + 4,
    n_embd=EMBEDDING_DIM,
    n_layer=2,
    n_head=6,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    bos_token_id=1,
    eos_token_id=2,
    pad_token_id=0,
)
gpt2 = GPT2LMHeadModel(gpt2_config)
gpt2.loss_type = "ForCausalLM"

model = ClipCaptionModel(mapper=mapper, gpt2=gpt2).to(DEVICE)

mapper_stats = model.mapper.count_parameters()
total_parameters = sum(parameter.numel() for parameter in model.parameters())

print("Mapper parameters:", mapper_stats["total_parameters"])
print("Total parameters :", total_parameters)
print("PASS: model dimensions are compatible")

## 6. Kiểm tra các tensor trước GPT-2

Training truyền `labels=batch["input_ids"]`. Hàm chuẩn bị phải tạo ba tensor có cùng sequence length `P + L`.

In [ ]:
prepared = model.prepare_gpt2_inputs(
    image_embed=batch["image_embed"],
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["input_ids"],
)

inputs_embeds = prepared["inputs_embeds"]
extended_attention_mask = prepared["attention_mask"]
extended_labels = prepared["labels"]
sequence_length = PREFIX_LENGTH + CAPTION_LENGTH

assert inputs_embeds.shape == (BATCH_SIZE, sequence_length, EMBEDDING_DIM)
assert extended_attention_mask.shape == (BATCH_SIZE, sequence_length)
assert extended_labels.shape == (BATCH_SIZE, sequence_length)
assert torch.all(extended_attention_mask[:, :PREFIX_LENGTH] == 1)
assert torch.all(extended_labels[:, :PREFIX_LENGTH] == -100)
assert torch.all(
    extended_labels[:, PREFIX_LENGTH:][batch["attention_mask"] == 0]
    == -100
)
assert torch.equal(
    extended_labels[:, PREFIX_LENGTH:][batch["attention_mask"] == 1],
    batch["input_ids"][batch["attention_mask"] == 1],
)

print("inputs_embeds shape          :", tuple(inputs_embeds.shape))
print("extended attention mask shape:", tuple(extended_attention_mask.shape))
print("extended labels shape       :", tuple(extended_labels.shape))
print("PASS: GPT-2 input contract")

## 7. Training forward và gradient

GPT-2 tự shift logits và labels cho causal language modeling. Không shift labels thủ công. Loss phải hữu hạn và gradient phải truyền qua GPT-2 về Mapper.

In [ ]:
model.train()
model.zero_grad(set_to_none=True)

training_outputs = model(
    image_embed=batch["image_embed"],
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["input_ids"],
)

assert training_outputs.loss is not None
assert torch.isfinite(training_outputs.loss)
assert training_outputs.logits.shape == (
    BATCH_SIZE,
    PREFIX_LENGTH + CAPTION_LENGTH,
    VOCAB_SIZE,
)

training_outputs.loss.backward()

mapper_gradients = [
    parameter.grad
    for parameter in model.mapper.parameters()
    if parameter.requires_grad
]
assert mapper_gradients
assert all(gradient is not None for gradient in mapper_gradients)
assert all(torch.isfinite(gradient).all() for gradient in mapper_gradients)
assert any(gradient.abs().sum() > 0 for gradient in mapper_gradients)

print("Training loss:", training_outputs.loss.item())
print("Logits shape :", tuple(training_outputs.logits.shape))
print("PASS: loss sends gradient to Mapper")

## 8. Freeze GPT-2 nhưng vẫn train Mapper

Freeze GPT-2 bằng `requires_grad=False`. Không bọc GPT-2 forward trong `torch.no_grad()`, vì gradient vẫn cần đi xuyên qua GPT-2 để tới visual prefix và Mapper.

In [ ]:
for parameter in model.gpt2.parameters():
    parameter.requires_grad = False

model.zero_grad(set_to_none=True)
frozen_outputs = model(
    image_embed=batch["image_embed"],
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["input_ids"],
)
frozen_outputs.loss.backward()

mapper_gradients = [
    parameter.grad
    for parameter in model.mapper.parameters()
    if parameter.requires_grad
]
gpt2_gradients = [parameter.grad for parameter in model.gpt2.parameters()]

assert all(gradient is not None for gradient in mapper_gradients)
assert any(gradient.abs().sum() > 0 for gradient in mapper_gradients)
assert all(gradient is None for gradient in gpt2_gradients)

print("Frozen GPT-2 loss:", frozen_outputs.loss.item())
print("PASS: frozen GPT-2 propagates gradient to Mapper")

## 9. Validation loss và forward không có labels

`model.eval()` không đồng nghĩa với không tính loss. Validation vẫn truyền labels. Forward không có labels trả logits nhưng không có loss; đây là contract cần cho generation sau này.

In [ ]:
model.eval()

validation_outputs = model(
    image_embed=batch["image_embed"],
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["input_ids"],
)
assert validation_outputs.loss is not None
assert torch.isfinite(validation_outputs.loss)

no_label_outputs = model(
    image_embed=batch["image_embed"],
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
)
assert no_label_outputs.loss is None
assert no_label_outputs.logits.shape == validation_outputs.logits.shape

print("Validation loss:", validation_outputs.loss.item())
print("PASS: labels=None skips loss without depending on model.train/eval")

## 10. Phát hiện input không hợp lệ

Lỗi shape hoặc dtype nên được phát hiện trước khi đi sâu vào GPT-2 để thông báo dễ hiểu hơn.

In [ ]:
def expect_error(error_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")


expect_error(
    ValueError,
    model,
    batch["image_embed"],
    batch["input_ids"],
    batch["attention_mask"][:, :-1],
)
expect_error(
    ValueError,
    model,
    batch["image_embed"][:1],
    batch["input_ids"],
    batch["attention_mask"],
)
expect_error(
    TypeError,
    model,
    batch["image_embed"],
    batch["input_ids"].float(),
    batch["attention_mask"],
)
expect_error(
    ValueError,
    model,
    batch["image_embed"],
    batch["input_ids"],
    batch["attention_mask"],
    labels=batch["input_ids"][:, :-1],
)

print("PASS: invalid inputs are rejected with clear errors")

## 11. Kiến thức và tối ưu cần nhớ

### Không tạo labels trong Dataset

Labels là dữ liệu dẫn xuất từ token IDs và attention mask. Tạo tại model giúp Dataset gọn, generation không tạo dữ liệu thừa và tránh nhiều nguồn logic.

### Luôn lấy prefix length từ output thực tế

Dùng `prefix_embeddings.size(1)` thay vì hard-code `10`.

### Freeze GPT-2 trước khi fine-tune toàn bộ

Baseline nên freeze GPT-2 và chỉ train Mapper. Sau khi loss ổn định mới cân nhắc mở một số layer GPT-2 với learning rate nhỏ hơn.

### Mixed precision và batch size

Khi dùng pretrained GPT-2 thật, nên dùng autocast/mixed precision trên GPU và bắt đầu với batch size nhỏ. Đây là tối ưu lớn hơn nhiều so với thay đổi cách nối mask.

### Generation chưa nằm trong notebook này

Forward với `labels=None` chỉ kiểm tra contract không-loss. Greedy/beam generation cần một pipeline decoding riêng vì attention mask và token sequence tăng sau mỗi bước.

## 12. Checklist trước khi chuyển thành `.py`

- [ ] Dataset chỉ trả `image_embed`, `input_ids`, `attention_mask`.
- [ ] Mapper dimension bằng GPT-2 embedding dimension.
- [ ] Prefix length lấy từ output của Mapper.
- [ ] `inputs_embeds`, extended mask và extended labels có cùng sequence length.
- [ ] Prefix mask bằng `1`.
- [ ] Prefix labels và padding labels bằng `-100`.
- [ ] Training và validation truyền labels.
- [ ] Forward không labels không tính loss.
- [ ] GPT-2 tự shift labels; code không shift thủ công.
- [ ] Loss hữu hạn và gradient truyền về Mapper.
- [ ] Freeze GPT-2 không dùng `torch.no_grad()`.
- [ ] Unit test dùng GPT-2 nhỏ, không phụ thuộc Internet.
- [ ] Production chỉ import theo cấu trúc mới `src.clipcap...`.

In [ ]:
print("All ClipCap integration checks completed successfully.")